## Lightweight LLM-Powered Equity Research Workflow

This notebook is a lab-style Google Colab project for building a lightweight equity research workflow with two clearly separated responsibilities:

> **Python calculates and structures the evidence.**  
> **The LLM synthesizes and explains the evidence.**

The notebook uses the **Financial Modeling Prep API** to collect company profile data, financial statements, valuation ratios, dividends, price history, and news. Python then calculates deterministic financial evidence such as rolling trailing-twelve-month revenue, margins, free cash flow, leverage ratios, volatility, drawdowns, and relative performance versus SPY.

That structured evidence is converted into JSON and sent to the **OpenAI Responses API**, where the model writes a balanced equity research note.

### What this project is not

This is **not semantic RAG**. It does not use embeddings, vector databases, document retrieval, or similarity search. It is better described as:

> **Deterministic evidence routing + LLM synthesis over structured financial evidence.**

The LLM is not asked to calculate financial metrics from raw financial statements. Instead, the notebook calculates the metrics directly in Python and asks the LLM to interpret the already-computed evidence.

### Important disclaimer

This notebook is for educational and research workflow demonstration purposes only. It does **not** provide financial advice, investment advice, or buy/sell/hold recommendations.


# 1. Install Needed Libraries

In [ ]:
%pip install -q openai pandas numpy requests tqdm

# 2. Import Libraries


In [ ]:
import json
import math
import time
import requests
import numpy as np
import pandas as pd

from datetime import date, datetime, timedelta
from typing import Any, Optional
from tqdm.auto import tqdm
from openai import OpenAI
from google.colab import userdata
from IPython.display import Markdown, display


# 3. Load API Keys from Colab Secrets

In Google Colab, add two secrets:

- `FMP_API_KEY`
- `OPENAI_API_KEY`

Then give this notebook access to those secrets from the Colab sidebar.


In [ ]:
FMP_API_KEY = userdata.get("FMPAPIKEY")
OPENAI_API_KEY = userdata.get("OpenAIKey")

missing_keys = []
if not FMP_API_KEY:
    missing_keys.append("FMP_API_KEY")
if not OPENAI_API_KEY:
    missing_keys.append("OPENAI_API_KEY")

if missing_keys:
    raise ValueError(
        "Missing required Colab secret(s): " + ", ".join(missing_keys) +
        ". Add them in Colab Secrets before running the notebook."
    )

client = OpenAI(api_key=OPENAI_API_KEY)
print("API keys loaded successfully from Colab Secrets.")


API keys loaded successfully from Colab Secrets.


# 4. Define Global Settings


In [ ]:
FMP_BASE_URL = "https://financialmodelingprep.com/stable"
# OPENAI_MODEL = "gpt-4.1-mini"
OPENAI_MODEL = "gpt-5"

BENCHMARK_SYMBOL = "SPY"

# User has access up to 750 FMP requests per minute.
# This notebook is intentionally simple and makes a modest number of requests per ticker,
# so heavy rate limiting is not necessary for the lab workflow.
DEFAULT_QUARTERLY_LIMIT = 24  # enough for approximately five years of rolling TTM windows
DEFAULT_PRICE_HISTORY_YEARS = 5
NEWS_LIMIT_STOCK = 25
NEWS_LIMIT_GENERAL = 25

TODAY = date.today()


# 5. Define FMP Request Helper

The helper below centralizes the FMP API call pattern. It adds the API key, raises readable errors, and returns parsed JSON.


In [ ]:
def fmp_get(endpoint: str, params: Optional[dict] = None) -> Any:
    """GET a Financial Modeling Prep stable endpoint and return parsed JSON.

    Parameters
    ----------
    endpoint:
        Endpoint path without the base URL, e.g. "profile" or "/profile".
    params:
        Query parameters. The API key is added automatically.

    Returns
    -------
    Parsed JSON response.
    """
    params = dict(params or {})
    params["apikey"] = FMP_API_KEY

    endpoint = endpoint.strip()
    if endpoint.startswith("/"):
        endpoint = endpoint[1:]

    url = f"{FMP_BASE_URL}/{endpoint}"
    response = requests.get(url, params=params, timeout=30)

    if not response.ok:
        raise RuntimeError(
            f"FMP request failed: HTTP {response.status_code}\n"
            f"URL: {response.url}\n"
            f"Response: {response.text[:500]}"
        )

    try:
        data = response.json()
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"FMP returned non-JSON response for {response.url}") from exc

    # FMP sometimes returns error messages inside a successful JSON payload.
    if isinstance(data, dict) and any(k.lower() in {"error", "error message"} for k in data.keys()):
        raise RuntimeError(f"FMP returned an error payload: {data}")

    return data


def safe_fmp_get(endpoint: str, params: Optional[dict] = None, default: Any = None, label: Optional[str] = None) -> Any:
    """Notebook-friendly optional FMP request.

    Required data should use fmp_get directly. Optional data can use this helper so
    endpoint coverage differences do not break the whole notebook.
    """
    try:
        return fmp_get(endpoint, params=params)
    except Exception as exc:
        name = label or endpoint
        print(f"Warning: optional FMP request failed for {name}: {exc}")
        return default


# 6. Define Custom FMP Endpoint Functions

These functions keep endpoint details out of the main evidence-building workflow.


In [ ]:
def _first_item(data: Any) -> dict:
    """Return the first dictionary from an FMP response when appropriate."""
    if isinstance(data, list) and data:
        return data[0] if isinstance(data[0], dict) else {}
    if isinstance(data, dict):
        return data
    return {}


def get_company_profile(symbol: str) -> dict:
    data = fmp_get("profile", {"symbol": symbol})
    return _first_item(data)


def get_income_statement_quarterly(symbol: str, limit: int = DEFAULT_QUARTERLY_LIMIT) -> list[dict]:
    return fmp_get("income-statement", {"symbol": symbol, "period": "quarter", "limit": limit})


def get_balance_sheet_quarterly(symbol: str, limit: int = DEFAULT_QUARTERLY_LIMIT) -> list[dict]:
    return fmp_get("balance-sheet-statement", {"symbol": symbol, "period": "quarter", "limit": limit})


def get_cash_flow_quarterly(symbol: str, limit: int = DEFAULT_QUARTERLY_LIMIT) -> list[dict]:
    return fmp_get("cash-flow-statement", {"symbol": symbol, "period": "quarter", "limit": limit})


def get_ttm_ratios(symbol: str) -> dict:
    data = safe_fmp_get("ratios-ttm", {"symbol": symbol}, default=[], label="ratios-ttm")
    return _first_item(data)


def get_key_metrics_ttm(symbol: str) -> dict:
    # Included as optional evidence. Endpoint coverage can vary by plan/symbol.
    data = safe_fmp_get("key-metrics-ttm", {"symbol": symbol}, default=[], label="key-metrics-ttm")
    return _first_item(data)


def get_historical_ratios(symbol: str, period: str = "quarter", limit: int = DEFAULT_QUARTERLY_LIMIT) -> list[dict]:
    return safe_fmp_get(
        "ratios",
        {"symbol": symbol, "period": period, "limit": limit},
        default=[],
        label="historical ratios"
    ) or []


def get_historical_key_metrics(symbol: str, period: str = "quarter", limit: int = DEFAULT_QUARTERLY_LIMIT) -> list[dict]:
    return safe_fmp_get(
        "key-metrics",
        {"symbol": symbol, "period": period, "limit": limit},
        default=[],
        label="historical key metrics"
    ) or []


def get_historical_ratios_or_key_metrics(symbol: str, period: str = "quarter", limit: int = DEFAULT_QUARTERLY_LIMIT) -> dict:
    """Fetch both historical ratios and key metrics in one notebook-friendly call."""
    return {
        "historical_ratios": get_historical_ratios(symbol, period=period, limit=limit),
        "historical_key_metrics": get_historical_key_metrics(symbol, period=period, limit=limit),
    }


def get_price_history(symbol: str, years: int = DEFAULT_PRICE_HISTORY_YEARS) -> list[dict]:
    end_date = date.today()
    start_date = end_date - timedelta(days=int(years * 365.25) + 10)
    return fmp_get(
        "historical-price-eod/full",
        {
            "symbol": symbol,
            "from": start_date.isoformat(),
            "to": end_date.isoformat(),
        },
    )


def get_dividend_history(symbol: str, years: int = DEFAULT_PRICE_HISTORY_YEARS) -> list[dict]:
    end_date = date.today()
    start_date = end_date - timedelta(days=int(years * 365.25) + 10)
    data = safe_fmp_get(
        "dividends",
        {
            "symbol": symbol,
            "from": start_date.isoformat(),
            "to": end_date.isoformat(),
        },
        default=[],
        label="dividends"
    )
    return data or []


def get_stock_news(symbol: str, limit: int = NEWS_LIMIT_STOCK) -> list[dict]:
    # FMP's search stock news endpoint accepts symbols, plus page/limit.
    data = safe_fmp_get(
        "news/stock",
        {"symbols": symbol, "page": 0, "limit": limit},
        default=[],
        label="stock news"
    )
    return data or []


def get_general_market_news(limit: int = NEWS_LIMIT_GENERAL) -> list[dict]:
    data = safe_fmp_get(
        "news/general-latest",
        {"page": 0, "limit": limit},
        default=[],
        label="general market news"
    )
    return data or []


# 7. Data Cleaning Helpers


In [ ]:
def to_dataframe(records: Any) -> pd.DataFrame:
    """Convert a list/dict FMP payload to a pandas DataFrame."""
    if records is None:
        return pd.DataFrame()
    if isinstance(records, dict):
        # Some legacy endpoints wrap rows under a 'historical' key.
        if "historical" in records and isinstance(records["historical"], list):
            records = records["historical"]
        else:
            records = [records]
    if not isinstance(records, list):
        return pd.DataFrame()
    return pd.DataFrame(records)


def convert_date_column(df: pd.DataFrame, column: str = "date") -> pd.DataFrame:
    df = df.copy()
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


def sort_by_date(df: pd.DataFrame, ascending: bool = True, column: str = "date") -> pd.DataFrame:
    df = convert_date_column(df, column=column)
    if column in df.columns:
        df = df.dropna(subset=[column]).sort_values(column, ascending=ascending).reset_index(drop=True)
    return df

def coerce_numeric_columns(df: pd.DataFrame, exclude: Optional[list[str]] = None) -> pd.DataFrame:
    """Convert fully numeric-looking columns while leaving text columns unchanged.

    pandas deprecated pd.to_numeric(..., errors="ignore"), so this function
    now attempts strict conversion and keeps the original column if conversion
    is not valid for the full column.
    """
    df = df.copy()
    exclude = set(exclude or [])

    for col in df.columns:
        if col in exclude:
            continue

        try:
            df[col] = pd.to_numeric(df[col], errors="raise")
        except (ValueError, TypeError):
            # Leave non-numeric or mixed text columns unchanged.
            pass

    return df


def standardize_price_df(records: Any) -> pd.DataFrame:
    """Create a canonical close_for_returns column from common FMP price fields."""
    df = to_dataframe(records)
    if df.empty:
        return df

    df = sort_by_date(df, ascending=True)
    df = coerce_numeric_columns(df, exclude=["date", "symbol", "label"])

    close_candidates = ["adjClose", "adj_close", "adjustedClose", "close"]
    selected = None
    for col in close_candidates:
        if col in df.columns and df[col].notna().any():
            selected = col
            break

    if selected is None:
        raise ValueError(
            "Could not find a usable price column. Expected one of: " + ", ".join(close_candidates)
        )

    df["close_for_returns"] = pd.to_numeric(df[selected], errors="coerce")
    df = df.dropna(subset=["date", "close_for_returns"]).reset_index(drop=True)
    return df


def clean_statement_df(records: Any) -> pd.DataFrame:
    df = to_dataframe(records)
    if df.empty:
        return df
    df = sort_by_date(df, ascending=True)
    df = coerce_numeric_columns(df, exclude=["date", "symbol", "period", "calendarYear", "reportedCurrency", "cik", "fillingDate", "acceptedDate", "link", "finalLink"])
    return df


def safe_number(value: Any) -> Optional[float]:
    if value is None:
        return None
    try:
        x = float(value)
        if math.isnan(x) or math.isinf(x):
            return None
        return x
    except Exception:
        return None


def safe_divide(numerator: Any, denominator: Any) -> Optional[float]:
    numerator = safe_number(numerator)
    denominator = safe_number(denominator)
    if numerator is None or denominator in (None, 0):
        return None
    return numerator / denominator


def pick_value(row: Any, candidates: list[str]) -> Optional[float]:
    """Pick the first non-null numeric value from a row-like object."""
    if row is None:
        return None
    for col in candidates:
        if col in row:
            value = safe_number(row[col])
            if value is not None:
                return value
    return None


def latest_record(df: pd.DataFrame) -> dict:
    if df.empty:
        return {}
    return df.sort_values("date", ascending=False).iloc[0].to_dict()


def pct(value: Any) -> Optional[float]:
    value = safe_number(value)
    return None if value is None else value * 100


def compact_round(value: Any, digits: int = 4) -> Any:
    value = safe_number(value)
    if value is None:
        return None
    return round(value, digits)


# 8. TTM Financial Metric Helper Functions

This is the core deterministic-financial-evidence section. We use quarterly financial statements to build rolling trailing-twelve-month values.

For each TTM period, Python sums the latest four quarters of income statement and cash flow fields, then uses the balance sheet snapshot at the period end where relevant.


In [ ]:
FIELD_ALIASES = {
    "revenue": ["revenue", "totalRevenue"],
    "gross_profit": ["grossProfit"],
    "operating_income": ["operatingIncome"],
    "net_income": ["netIncome"],
    "operating_cash_flow": ["operatingCashFlow", "cashFlowFromOperatingActivities", "netCashProvidedByOperatingActivities"],
    "capital_expenditure": ["capitalExpenditure", "capitalExpenditures", "investmentsInPropertyPlantAndEquipment"],
    "cash": ["cashAndCashEquivalents", "cashAndShortTermInvestments", "cashAndCashEquivalentsAndShortTermInvestments"],
    "total_debt": ["totalDebt", "shortTermDebtAndLongTermDebtTotal", "shortTermDebtAndLongTermDebt"],
    "short_term_debt": ["shortTermDebt"],
    "long_term_debt": ["longTermDebt"],
    "total_equity": ["totalStockholdersEquity", "totalEquity", "totalShareholderEquity"],
    "total_assets": ["totalAssets"],
}


def calculate_free_cash_flow(operating_cash_flow: Any, capital_expenditure: Any) -> Optional[float]:
    """Calculate FCF while handling FMP capex sign conventions.

    FMP often reports capitalExpenditure as a negative number. If capex is negative,
    FCF = OCF + capex. If capex is positive, FCF = OCF - capex.
    """
    ocf = safe_number(operating_cash_flow)
    capex = safe_number(capital_expenditure)
    if ocf is None or capex is None:
        return None
    return ocf + capex if capex < 0 else ocf - capex


def _sum_field(window: pd.DataFrame, candidates: list[str]) -> Optional[float]:
    for col in candidates:
        if col in window.columns:
            values = pd.to_numeric(window[col], errors="coerce").dropna()
            if len(values) > 0:
                return float(values.sum())
    return None


def build_ttm_series(income_df: pd.DataFrame, cash_flow_df: pd.DataFrame, balance_df: pd.DataFrame) -> pd.DataFrame:
    """Build rolling TTM fundamentals from quarterly financial statements."""
    income_df = sort_by_date(income_df, ascending=True)
    cash_flow_df = sort_by_date(cash_flow_df, ascending=True)
    balance_df = sort_by_date(balance_df, ascending=True)

    if income_df.empty or len(income_df) < 4:
        return pd.DataFrame()

    cf_by_date = cash_flow_df.set_index("date") if "date" in cash_flow_df.columns and not cash_flow_df.empty else pd.DataFrame()
    bs_by_date = balance_df.set_index("date") if "date" in balance_df.columns and not balance_df.empty else pd.DataFrame()

    rows = []
    for idx in range(3, len(income_df)):
        income_window = income_df.iloc[idx - 3: idx + 1]
        period_end = income_df.iloc[idx]["date"]

        revenue = _sum_field(income_window, FIELD_ALIASES["revenue"])
        gross_profit = _sum_field(income_window, FIELD_ALIASES["gross_profit"])
        operating_income = _sum_field(income_window, FIELD_ALIASES["operating_income"])
        net_income = _sum_field(income_window, FIELD_ALIASES["net_income"])

        cf_window = pd.DataFrame()
        if not cf_by_date.empty:
            dates = list(income_window["date"])
            cf_window = cash_flow_df[cash_flow_df["date"].isin(dates)]

        operating_cash_flow = _sum_field(cf_window, FIELD_ALIASES["operating_cash_flow"]) if not cf_window.empty else None
        capex = _sum_field(cf_window, FIELD_ALIASES["capital_expenditure"]) if not cf_window.empty else None
        free_cash_flow = calculate_free_cash_flow(operating_cash_flow, capex)

        bs_row = None
        if not bs_by_date.empty and period_end in bs_by_date.index:
            candidate = bs_by_date.loc[period_end]
            bs_row = candidate.iloc[0] if isinstance(candidate, pd.DataFrame) else candidate

        cash = pick_value(bs_row, FIELD_ALIASES["cash"])
        total_debt = pick_value(bs_row, FIELD_ALIASES["total_debt"])
        if total_debt is None:
            std = pick_value(bs_row, FIELD_ALIASES["short_term_debt"]) or 0
            ltd = pick_value(bs_row, FIELD_ALIASES["long_term_debt"]) or 0
            total_debt = std + ltd if (std or ltd) else None
        total_equity = pick_value(bs_row, FIELD_ALIASES["total_equity"])
        total_assets = pick_value(bs_row, FIELD_ALIASES["total_assets"])

        rows.append({
            "period_end": period_end,
            "ttm_revenue": revenue,
            "ttm_gross_profit": gross_profit,
            "ttm_operating_income": operating_income,
            "ttm_net_income": net_income,
            "ttm_operating_cash_flow": operating_cash_flow,
            "ttm_capex": capex,
            "ttm_free_cash_flow": free_cash_flow,
            "ttm_gross_margin": safe_divide(gross_profit, revenue),
            "ttm_operating_margin": safe_divide(operating_income, revenue),
            "ttm_net_margin": safe_divide(net_income, revenue),
            "cash_and_equivalents": cash,
            "total_debt": total_debt,
            "total_equity": total_equity,
            "total_assets": total_assets,
            "debt_to_equity": safe_divide(total_debt, total_equity),
            "debt_to_assets": safe_divide(total_debt, total_assets),
            "net_debt": None if total_debt is None or cash is None else total_debt - cash,
            "cash_to_debt": safe_divide(cash, total_debt),
        })

    return pd.DataFrame(rows).sort_values("period_end", ascending=True).reset_index(drop=True)


def calculate_growth_series(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return values.pct_change().replace([np.inf, -np.inf], np.nan)


def calculate_average_ttm_growth(ttm_df: pd.DataFrame, column: str) -> Optional[float]:
    if ttm_df.empty or column not in ttm_df.columns:
        return None
    growth = calculate_growth_series(ttm_df[column]).dropna()
    if growth.empty:
        return None
    return float(growth.mean())


def classify_trend(series: Any, tolerance: float = 0.05) -> str:
    """Classify a numeric series as expanding, contracting, stable, or insufficient data."""
    if series is None:
        return "insufficient data"

    s = pd.Series(series).dropna().astype(float)
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 6:
        return "insufficient data"

    recent = s.iloc[-3:].mean()
    older = s.iloc[-6:-3].mean()

    if older == 0 or pd.isna(older) or pd.isna(recent):
        return "insufficient data"

    change = (recent - older) / abs(older)
    if change > tolerance:
        return "expanding"
    if change < -tolerance:
        return "contracting"
    return "stable"


def calculate_ttm_fundamentals_summary(ttm_df: pd.DataFrame) -> dict:
    if ttm_df.empty:
        return {
            "status": "insufficient data",
            "reason": "Fewer than four quarters of usable financial statements were available."
        }

    latest = ttm_df.iloc[-1]

    return {
        "revenue": {
            "latest_ttm_revenue": compact_round(latest.get("ttm_revenue"), 2),
            "trend": classify_trend(ttm_df["ttm_revenue"]),
            "average_ttm_revenue_growth": compact_round(calculate_average_ttm_growth(ttm_df, "ttm_revenue"), 4),
        },
        "margins": {
            "latest_ttm_gross_margin": compact_round(latest.get("ttm_gross_margin"), 4),
            "latest_ttm_operating_margin": compact_round(latest.get("ttm_operating_margin"), 4),
            "latest_ttm_net_margin": compact_round(latest.get("ttm_net_margin"), 4),
            "gross_margin_trend": classify_trend(ttm_df["ttm_gross_margin"]),
            "operating_margin_trend": classify_trend(ttm_df["ttm_operating_margin"]),
            "net_margin_trend": classify_trend(ttm_df["ttm_net_margin"]),
        },
        "free_cash_flow": {
            "latest_ttm_operating_cash_flow": compact_round(latest.get("ttm_operating_cash_flow"), 2),
            "latest_ttm_free_cash_flow": compact_round(latest.get("ttm_free_cash_flow"), 2),
            "free_cash_flow_trend": classify_trend(ttm_df["ttm_free_cash_flow"]),
            "average_ttm_free_cash_flow_growth": compact_round(calculate_average_ttm_growth(ttm_df, "ttm_free_cash_flow"), 4),
        },
        "ttm_periods_available": int(len(ttm_df)),
        "latest_ttm_period_end": str(latest.get("period_end")) if pd.notna(latest.get("period_end")) else None,
    }


## Convenience TTM Accessor Functions

These small functions expose the main TTM values directly for readers who want to inspect one metric at a time.


In [ ]:
def _latest_ttm_value(ttm_df: pd.DataFrame, column: str) -> Optional[float]:
    if ttm_df.empty or column not in ttm_df.columns:
        return None
    return safe_number(ttm_df.iloc[-1].get(column))


def calculate_ttm_revenue(ttm_df: pd.DataFrame) -> Optional[float]:
    return _latest_ttm_value(ttm_df, "ttm_revenue")


def calculate_ttm_gross_profit(ttm_df: pd.DataFrame) -> Optional[float]:
    return _latest_ttm_value(ttm_df, "ttm_gross_profit")


def calculate_ttm_operating_income(ttm_df: pd.DataFrame) -> Optional[float]:
    return _latest_ttm_value(ttm_df, "ttm_operating_income")


def calculate_ttm_net_income(ttm_df: pd.DataFrame) -> Optional[float]:
    return _latest_ttm_value(ttm_df, "ttm_net_income")


def calculate_ttm_operating_cash_flow(ttm_df: pd.DataFrame) -> Optional[float]:
    return _latest_ttm_value(ttm_df, "ttm_operating_cash_flow")


def calculate_ttm_free_cash_flow_latest(ttm_df: pd.DataFrame) -> Optional[float]:
    return _latest_ttm_value(ttm_df, "ttm_free_cash_flow")


def calculate_ttm_margins(ttm_df: pd.DataFrame) -> dict:
    return {
        "gross_margin": _latest_ttm_value(ttm_df, "ttm_gross_margin"),
        "operating_margin": _latest_ttm_value(ttm_df, "ttm_operating_margin"),
        "net_margin": _latest_ttm_value(ttm_df, "ttm_net_margin"),
    }


# 9. Valuation Trend Helper Functions

The latest valuation ratios come from FMP's TTM ratios endpoint when available. Historical trends use quarterly ratios and key metrics when the relevant fields are present.


In [ ]:
VALUATION_ALIASES = {
    "pe": ["priceToEarningsRatioTTM", "priceEarningsRatio", "peRatioTTM", "peRatio", "priceToEarningsRatio"],
    "pb": ["priceToBookRatioTTM", "priceToBookRatio", "pbRatioTTM", "pbRatio"],
    "ps": ["priceToSalesRatioTTM", "priceToSalesRatio", "priceSalesRatioTTM", "priceSalesRatio"],
}


def pick_metric_from_dict(data: dict, candidates: list[str]) -> Optional[float]:
    if not isinstance(data, dict):
        return None
    return pick_value(data, candidates)


def extract_metric_series(records: list[dict], candidates: list[str]) -> pd.Series:
    df = clean_statement_df(records)
    if df.empty:
        return pd.Series(dtype=float)

    for col in candidates:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce").dropna()
            if not s.empty:
                return s.reset_index(drop=True)

    return pd.Series(dtype=float)


def build_valuation_evidence(ttm_ratios: dict, historical_ratios: list[dict], historical_key_metrics: list[dict]) -> dict:
    evidence = {}

    for label, aliases in VALUATION_ALIASES.items():
        latest = pick_metric_from_dict(ttm_ratios, aliases)

        hist_series = extract_metric_series(historical_ratios, aliases)
        if hist_series.empty:
            hist_series = extract_metric_series(historical_key_metrics, aliases)

        evidence[label] = {
            f"latest_{label}": compact_round(latest, 4),
            f"{label}_trend": classify_trend(hist_series),
            f"{label}_average": compact_round(hist_series.mean(), 4) if not hist_series.empty else None,
            "historical_observations": int(hist_series.dropna().shape[0]) if not hist_series.empty else 0,
        }

    return evidence


# 10. Balance Sheet and Debt Ratio Helpers


In [ ]:
def build_balance_sheet_debt_evidence(ttm_df: pd.DataFrame, balance_df: pd.DataFrame) -> dict:
    if ttm_df.empty:
        return {
            "status": "insufficient data",
            "reason": "No TTM balance sheet series was available."
        }

    latest = ttm_df.iloc[-1]

    return {
        "latest_cash_and_equivalents": compact_round(latest.get("cash_and_equivalents"), 2),
        "latest_total_debt": compact_round(latest.get("total_debt"), 2),
        "latest_total_equity": compact_round(latest.get("total_equity"), 2),
        "latest_total_assets": compact_round(latest.get("total_assets"), 2),
        "latest_debt_to_equity": compact_round(latest.get("debt_to_equity"), 4),
        "debt_to_equity_trend": classify_trend(ttm_df["debt_to_equity"]),
        "latest_debt_to_assets": compact_round(latest.get("debt_to_assets"), 4),
        "debt_to_assets_trend": classify_trend(ttm_df["debt_to_assets"]),
        "latest_net_debt": compact_round(latest.get("net_debt"), 2),
        "latest_cash_to_debt": compact_round(latest.get("cash_to_debt"), 4),
    }


# 11. Dividend History Helper


In [ ]:
def summarize_dividends(dividend_records: list[dict]) -> dict:
    df = to_dataframe(dividend_records)
    if df.empty:
        return {
            "pays_dividend": False,
            "recent_dividends": [],
            "dividend_frequency": "none detected",
            "dividend_growth_trend": "insufficient data",
            "total_dividends_last_year": 0,
        }

    df = sort_by_date(df, ascending=True)
    amount_col = None
    for col in ["adjDividend", "dividend", "amount"]:
        if col in df.columns:
            amount_col = col
            break

    if amount_col is None:
        return {
            "pays_dividend": False,
            "recent_dividends": [],
            "dividend_frequency": "insufficient data",
            "dividend_growth_trend": "insufficient data",
            "total_dividends_last_year": None,
        }

    df[amount_col] = pd.to_numeric(df[amount_col], errors="coerce")
    df = df.dropna(subset=["date", amount_col])
    if df.empty:
        return {
            "pays_dividend": False,
            "recent_dividends": [],
            "dividend_frequency": "none detected",
            "dividend_growth_trend": "insufficient data",
            "total_dividends_last_year": 0,
        }

    recent = df.tail(8).sort_values("date", ascending=False)

    frequency = "insufficient data"
    if len(df) >= 3:
        day_diffs = df["date"].diff().dt.days.dropna()
        avg_days = day_diffs.mean()
        if pd.notna(avg_days):
            if avg_days <= 45:
                frequency = "monthly or more frequent"
            elif avg_days <= 120:
                frequency = "quarterly"
            elif avg_days <= 240:
                frequency = "semiannual"
            elif avg_days <= 420:
                frequency = "annual"
            else:
                frequency = "irregular"

    one_year_ago = pd.Timestamp(date.today() - timedelta(days=365))
    total_last_year = df.loc[df["date"] >= one_year_ago, amount_col].sum()

    annual = df.assign(year=df["date"].dt.year).groupby("year")[amount_col].sum().sort_index()
    growth_trend = classify_trend(annual) if len(annual) >= 6 else "insufficient data"

    recent_dividends = [
        {
            "date": str(row["date"].date()) if pd.notna(row["date"]) else None,
            "amount": compact_round(row[amount_col], 4),
        }
        for _, row in recent.iterrows()
    ]

    return {
        "pays_dividend": bool(len(df) > 0 and df[amount_col].sum() > 0),
        "recent_dividends": recent_dividends,
        "dividend_frequency": frequency,
        "dividend_growth_trend": growth_trend,
        "total_dividends_last_year": compact_round(total_last_year, 4),
    }


# 12. Market Performance Helper Functions

The notebook compares the selected stock to SPY using adjusted close when available.


In [ ]:
def calculate_returns(price_df: pd.DataFrame) -> pd.Series:
    if price_df.empty or "close_for_returns" not in price_df.columns:
        return pd.Series(dtype=float)
    returns = price_df["close_for_returns"].pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    return returns


def calculate_total_return(price_df: pd.DataFrame) -> Optional[float]:
    if price_df.empty or "close_for_returns" not in price_df.columns or len(price_df) < 2:
        return None
    start = price_df["close_for_returns"].iloc[0]
    end = price_df["close_for_returns"].iloc[-1]
    ratio = safe_divide(end, start)
    return ratio - 1 if ratio is not None else None


def calculate_annualized_return(price_df: pd.DataFrame) -> Optional[float]:
    if price_df.empty or len(price_df) < 2:
        return None
    total_return = calculate_total_return(price_df)
    if total_return is None:
        return None
    days = (price_df["date"].iloc[-1] - price_df["date"].iloc[0]).days
    if days <= 0:
        return None
    years = days / 365.25
    return (1 + total_return) ** (1 / years) - 1 if total_return > -1 else None


def calculate_annualized_volatility(price_df: pd.DataFrame) -> Optional[float]:
    returns = calculate_returns(price_df)
    if returns.empty:
        return None
    return float(returns.std() * np.sqrt(252))


def calculate_max_drawdown(price_df: pd.DataFrame) -> Optional[float]:
    if price_df.empty or "close_for_returns" not in price_df.columns:
        return None
    prices = price_df["close_for_returns"].dropna()
    if prices.empty:
        return None
    running_max = prices.cummax()
    drawdown = prices / running_max - 1
    return float(drawdown.min())


def _performance_block(price_df: pd.DataFrame) -> dict:
    return {
        "total_return": compact_round(calculate_total_return(price_df), 4),
        "annualized_return": compact_round(calculate_annualized_return(price_df), 4),
        "annualized_volatility": compact_round(calculate_annualized_volatility(price_df), 4),
        "max_drawdown": compact_round(calculate_max_drawdown(price_df), 4),
        "observations": int(len(price_df)),
        "start_date": str(price_df["date"].iloc[0].date()) if not price_df.empty else None,
        "end_date": str(price_df["date"].iloc[-1].date()) if not price_df.empty else None,
    }


def calculate_market_performance(stock_prices: pd.DataFrame, spy_prices: pd.DataFrame) -> dict:
    stock_prices = sort_by_date(stock_prices, ascending=True)
    spy_prices = sort_by_date(spy_prices, ascending=True)

    stock_last_252 = stock_prices.tail(252)
    spy_last_252 = spy_prices.tail(252)

    stock_multi = _performance_block(stock_prices)
    spy_multi = _performance_block(spy_prices)
    stock_252 = _performance_block(stock_last_252)
    spy_252 = _performance_block(spy_last_252)

    return {
        "multi_year": {
            "stock": stock_multi,
            "benchmark": spy_multi,
            "relative_annualized_return": compact_round(
                stock_multi["annualized_return"] - spy_multi["annualized_return"], 4
            ) if stock_multi.get("annualized_return") is not None and spy_multi.get("annualized_return") is not None else None,
        },
        "last_252_trading_days": {
            "stock": stock_252,
            "benchmark": spy_252,
            "relative_return": compact_round(
                stock_252["total_return"] - spy_252["total_return"], 4
            ) if stock_252.get("total_return") is not None and spy_252.get("total_return") is not None else None,
            "relative_volatility": compact_round(
                stock_252["annualized_volatility"] - spy_252["annualized_volatility"], 4
            ) if stock_252.get("annualized_volatility") is not None and spy_252.get("annualized_volatility") is not None else None,
        }
    }


# 13. News Helper Functions

To keep the prompt compact, the notebook passes short news metadata and excerpts only—not full articles.


In [ ]:
def compact_news_item(item: dict) -> dict:
    return {
        "date": item.get("publishedDate") or item.get("date") or item.get("published_at"),
        "title": item.get("title"),
        "source": item.get("site") or item.get("publisher") or item.get("source"),
        "summary_or_excerpt": item.get("text") or item.get("summary") or item.get("snippet") or item.get("description"),
        "url": item.get("url") or item.get("link"),
        "symbols": item.get("symbols") or item.get("symbol"),
    }


def compact_news(records: list[dict], limit: int) -> list[dict]:
    if not records:
        return []
    compacted = []
    for item in records[:limit]:
        if isinstance(item, dict):
            compacted.append(compact_news_item(item))
    return compacted


# 14. Deterministic Signal Summary

This section converts calculated evidence into simple deterministic signal lists. These are not recommendations. They only summarize evidence that may support, weaken, or complicate an outperformance thesis.


In [ ]:
def _add_if(condition: bool, target: list[str], message: str) -> None:
    if condition:
        target.append(message)


def build_deterministic_signal_summary(evidence_partial: dict) -> dict:
    supports = []
    weakens = []
    uncertainties = []

    fundamentals = evidence_partial.get("fundamentals_ttm", {})
    valuation = evidence_partial.get("valuation", {})
    balance = evidence_partial.get("balance_sheet_and_debt", {})
    perf = evidence_partial.get("market_performance", {})

    revenue = fundamentals.get("revenue", {})
    margins = fundamentals.get("margins", {})
    fcf = fundamentals.get("free_cash_flow", {})

    rev_growth = revenue.get("average_ttm_revenue_growth")
    fcf_growth = fcf.get("average_ttm_free_cash_flow_growth")

    _add_if(rev_growth is not None and rev_growth > 0, supports, "Average rolling TTM revenue growth is positive.")
    _add_if(rev_growth is not None and rev_growth < 0, weakens, "Average rolling TTM revenue growth is negative.")
    _add_if(revenue.get("trend") == "expanding", supports, "TTM revenue trend is expanding.")
    _add_if(revenue.get("trend") == "contracting", weakens, "TTM revenue trend is contracting.")

    for margin_name in ["gross_margin", "operating_margin", "net_margin"]:
        trend = margins.get(f"{margin_name}_trend")
        readable = margin_name.replace("_", " ")
        _add_if(trend == "expanding", supports, f"{readable.title()} trend is expanding.")
        _add_if(trend == "contracting", weakens, f"{readable.title()} trend is contracting.")

    _add_if(fcf_growth is not None and fcf_growth > 0, supports, "Average rolling TTM free cash flow growth is positive.")
    _add_if(fcf_growth is not None and fcf_growth < 0, weakens, "Average rolling TTM free cash flow growth is negative.")
    _add_if(fcf.get("free_cash_flow_trend") == "expanding", supports, "TTM free cash flow trend is expanding.")
    _add_if(fcf.get("free_cash_flow_trend") == "contracting", weakens, "TTM free cash flow trend is contracting.")

    for multiple in ["pe", "pb", "ps"]:
        block = valuation.get(multiple, {})
        trend = block.get(f"{multiple}_trend")
        if trend == "expanding":
            weakens.append(f"{multiple.upper()} multiple trend is expanding, which may indicate higher expectations or valuation risk.")
        elif trend == "contracting":
            supports.append(f"{multiple.upper()} multiple trend is contracting, which may reduce valuation pressure if fundamentals remain intact.")
        elif trend == "insufficient data":
            uncertainties.append(f"Historical {multiple.upper()} trend data is insufficient.")

    dte = balance.get("latest_debt_to_equity")
    _add_if(dte is not None and dte < 0.5, supports, "Latest debt-to-equity is relatively low.")
    _add_if(dte is not None and dte > 2.0, weakens, "Latest debt-to-equity is elevated.")
    _add_if(balance.get("debt_to_equity_trend") == "expanding", weakens, "Debt-to-equity trend is rising.")
    _add_if(balance.get("debt_to_equity_trend") == "contracting", supports, "Debt-to-equity trend is declining.")

    last_252 = perf.get("last_252_trading_days", {})
    rel_return = last_252.get("relative_return")
    rel_vol = last_252.get("relative_volatility")

    _add_if(rel_return is not None and rel_return > 0, supports, "Stock outperformed SPY over the last 252 trading days.")
    _add_if(rel_return is not None and rel_return < 0, weakens, "Stock underperformed SPY over the last 252 trading days.")
    _add_if(rel_vol is not None and rel_vol > 0, weakens, "Stock volatility exceeded SPY volatility over the last 252 trading days.")
    _add_if(rel_vol is not None and rel_vol < 0, supports, "Stock volatility was lower than SPY volatility over the last 252 trading days.")

    stock_dd = last_252.get("stock", {}).get("max_drawdown")
    spy_dd = last_252.get("benchmark", {}).get("max_drawdown")
    _add_if(stock_dd is not None and spy_dd is not None and stock_dd < spy_dd, weakens, "Stock max drawdown was worse than SPY over the last 252 trading days.")
    _add_if(stock_dd is not None and spy_dd is not None and stock_dd > spy_dd, supports, "Stock max drawdown was less severe than SPY over the last 252 trading days.")

    if fundamentals.get("status") == "insufficient data":
        uncertainties.append("TTM fundamentals are incomplete.")
    if not evidence_partial.get("news", {}).get("stock_specific_news"):
        uncertainties.append("Stock-specific news evidence is unavailable or empty.")
    uncertainties.append("News context can be noisy and may not map cleanly to future equity performance.")
    uncertainties.append("This workflow does not include peer-relative valuation, analyst estimates, SEC filings, or earnings transcript analysis.")

    return {
        "supports_outperformance_thesis": supports,
        "weakens_outperformance_thesis": weakens,
        "key_uncertainties": uncertainties,
    }


# 15. Build the Deterministic Evidence JSON

`build_deterministic_evidence(symbol)` is the main evidence-building function. It fetches data, calculates deterministic metrics, summarizes signals, and returns a structured JSON-like dictionary.


In [ ]:
def build_company_overview(profile: dict) -> dict:
    return {
        "company_name": profile.get("companyName") or profile.get("company_name"),
        "symbol": profile.get("symbol"),
        "exchange": profile.get("exchange") or profile.get("exchangeShortName"),
        "industry": profile.get("industry"),
        "sector": profile.get("sector"),
        "country": profile.get("country"),
        "market_cap": profile.get("marketCap") or profile.get("mktCap"),
        "price": profile.get("price"),
        "beta": profile.get("beta"),
        "description": profile.get("description"),
        "website": profile.get("website"),
    }


def build_deterministic_evidence(symbol: str) -> dict:
    symbol = symbol.strip().upper()
    if not symbol:
        raise ValueError("Symbol cannot be empty.")

    print(f"Building deterministic evidence for {symbol}...")

    profile = get_company_profile(symbol)
    income_raw = get_income_statement_quarterly(symbol)
    balance_raw = get_balance_sheet_quarterly(symbol)
    cash_flow_raw = get_cash_flow_quarterly(symbol)
    ttm_ratios = get_ttm_ratios(symbol)
    key_metrics_ttm = get_key_metrics_ttm(symbol)
    historical_ratios = get_historical_ratios(symbol)
    historical_key_metrics = get_historical_key_metrics(symbol)
    price_raw = get_price_history(symbol)
    spy_price_raw = get_price_history(BENCHMARK_SYMBOL)
    dividend_raw = get_dividend_history(symbol)
    stock_news_raw = get_stock_news(symbol)
    general_news_raw = get_general_market_news()

    income_df = clean_statement_df(income_raw)
    balance_df = clean_statement_df(balance_raw)
    cash_flow_df = clean_statement_df(cash_flow_raw)
    price_df = standardize_price_df(price_raw)
    spy_price_df = standardize_price_df(spy_price_raw)

    ttm_df = build_ttm_series(income_df, cash_flow_df, balance_df)
    fundamentals_ttm = calculate_ttm_fundamentals_summary(ttm_df)
    valuation = build_valuation_evidence(ttm_ratios, historical_ratios, historical_key_metrics)
    balance_sheet_and_debt = build_balance_sheet_debt_evidence(ttm_df, balance_df)
    dividends = summarize_dividends(dividend_raw)
    market_performance = calculate_market_performance(price_df, spy_price_df)

    news = {
        "stock_specific_news": compact_news(stock_news_raw, NEWS_LIMIT_STOCK),
        "general_market_news": compact_news(general_news_raw, NEWS_LIMIT_GENERAL),
    }

    evidence = {
        "symbol": symbol,
        "benchmark_symbol": BENCHMARK_SYMBOL,
        "research_question": f"Does the evidence support a credible setup for {symbol} to outperform {BENCHMARK_SYMBOL}?",
        "company_overview": build_company_overview(profile),
        "valuation": valuation,
        "fundamentals_ttm": fundamentals_ttm,
        "balance_sheet_and_debt": balance_sheet_and_debt,
        "dividends": dividends,
        "market_performance": market_performance,
        "news": news,
        "deterministic_signal_summary": {},
        "methodology_note": (
            "This evidence was assembled deterministically from FMP data. "
            "No embeddings, vector database, semantic retrieval, or document retrieval were used. "
            "Python calculated the financial metrics before the evidence was sent to the LLM."
        ),
        "data_quality_notes": {
            "income_statement_quarters": int(len(income_df)),
            "balance_sheet_quarters": int(len(balance_df)),
            "cash_flow_quarters": int(len(cash_flow_df)),
            "ttm_periods": int(len(ttm_df)),
            "stock_price_observations": int(len(price_df)),
            "benchmark_price_observations": int(len(spy_price_df)),
            "ttm_ratios_available": bool(ttm_ratios),
            "key_metrics_ttm_available": bool(key_metrics_ttm),
        },
    }

    evidence["deterministic_signal_summary"] = build_deterministic_signal_summary(evidence)
    return evidence


# 16. Inspect the Evidence JSON

Run this cell to build the evidence package for Apple as the default example.

The printed JSON is intentionally truncated so the notebook remains readable.


In [ ]:
evidence = build_deterministic_evidence("NVDA")

print("Top-level evidence keys:")
print(list(evidence.keys()))

print("\nTruncated evidence JSON preview:")
print(json.dumps(evidence, indent=2, default=str)[:6000])

Building deterministic evidence for NVDA...
Top-level evidence keys:
['symbol', 'benchmark_symbol', 'research_question', 'company_overview', 'valuation', 'fundamentals_ttm', 'balance_sheet_and_debt', 'dividends', 'market_performance', 'news', 'deterministic_signal_summary', 'methodology_note', 'data_quality_notes']

Truncated evidence JSON preview:
{
  "symbol": "NVDA",
  "benchmark_symbol": "SPY",
  "research_question": "Does the evidence support a credible setup for NVDA to outperform SPY?",
  "company_overview": {
    "company_name": "NVIDIA Corporation",
    "symbol": "NVDA",
    "exchange": "NASDAQ",
    "industry": "Semiconductors",
    "sector": "Technology",
    "country": "US",
    "market_cap": 5398618690000,
    "price": 222.89,
    "beta": 2.244,
    "description": "NVIDIA Corporation provides graphics, and compute and networking solutions in the United States, Taiwan, China, and internationally. The company's Graphics segment offers GeForce GPUs for gaming and PCs, the GeF

# 17. Define the OpenAI Prompt and API Call

The prompt enforces the central design rule: the LLM may synthesize, weigh, and explain the evidence, but it should not calculate new financial metrics or make investment recommendations.


In [ ]:
def call_openai_with_evidence(evidence: dict) -> str:
    evidence_json = json.dumps(evidence, indent=2, default=str)

    system_prompt = """
You are an equity research assistant.
You synthesize structured financial evidence into balanced, professional equity research commentary.
You do not provide financial advice.
You do not give buy, sell, or hold recommendations.
You do not make hard predictions.
Use evidence-weighted language.
Use only the provided evidence package.
Do not calculate new financial metrics.
Do not infer facts that are not present in the evidence.
Identify uncertainties and data limitations.
Weigh conflicting signals fairly.
Write in a concise, professional equity research style.
""".strip()

    user_prompt = f"""
Research question:
{evidence.get("research_question")}

Structured deterministic evidence package:
```json
{evidence_json}
```

Write a balanced equity research note with the following requirements:
- 4 to 7 concise paragraphs.
- Discuss evidence supporting an outperformance thesis.
- Discuss evidence weakening an outperformance thesis.
- Discuss valuation, fundamentals, balance sheet/debt, dividends, market behavior, and news where relevant.
- Identify key uncertainties.
- End with a balanced evidence-weighted conclusion.
- Do not provide financial advice.
- Do not say buy, sell, or hold.
- Do not calculate new metrics; use only the metrics in the evidence package.
""".strip()

    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=system_prompt,
        input=user_prompt,
    )

    return response.output_text


# 18. Final End-to-End Function

`analyze_stock(symbol)` builds the deterministic evidence JSON and sends it to OpenAI for synthesis.


In [ ]:
def analyze_stock(symbol: str) -> str:
    evidence = build_deterministic_evidence(symbol)
    research_note = call_openai_with_evidence(evidence)
    return research_note

# 19. Final Output


In [ ]:
# Generate and display the final research note in Markdown.
# Uncomment to run after you have confirmed your API keys and evidence payload.
Symbol = "PLTR"

analysis = analyze_stock(Symbol)
display(Markdown(analysis))

Building deterministic evidence for PLTR...


Palantir Technologies sits at the center of the AI/software infrastructure theme, with a large market cap, high beta (1.521), and significant investor attention. The research question is whether there is a credible setup for PLTR to outperform SPY. Multi-year data show substantial historical outperformance versus the benchmark, but the most recent one-year window reflects underperformance and elevated volatility, placing more weight on the durability of fundamentals and the resilience of the valuation.

Fundamentals lean supportive. TTM revenue is expanding at $5.224 billion, with average rolling TTM revenue growth positive. Profitability is a bright spot: gross margin is high (0.8407) and stable, while operating (0.3813) and net margins (0.4367) are expanding. Cash generation is strong, with TTM operating cash flow of $2.724 billion and free cash flow of $2.689 billion; free cash flow is expanding with positive average growth. These trends indicate a business scaling revenue and margins while compounding cash flow—key ingredients for sustained relative performance if maintained.

The balance sheet provides a cushion for execution and macro risk. Debt metrics are conservative (debt-to-equity 0.0251, contracting; debt-to-assets 0.0208, contracting), and cash-to-debt is 10.81, with net cash of approximately $2.08 billion. This low-leverage profile can support continued investment in growth and product development without immediate financing pressure. The company does not pay a dividend, so prospective returns hinge on earnings and cash flow compounding and market multiple behavior rather than cash distributions.

Valuation is the core counterweight. The latest P/E (159.79) is elevated, though its trend is contracting, which could reduce pressure if fundamentals keep improving. However, both P/B (43.15) and P/S (66.93) are in expanding trends, signaling rising expectations and valuation risk. At these levels, sentiment shifts or growth hiccups could translate into outsized drawdowns, even if the longer-term story remains intact.

Market behavior is mixed. Over the multi-year period since 2021, PLTR’s annualized return (0.4804) exceeded SPY (0.1255), with a sizable relative annualized return (0.3549). By contrast, over the last 252 trading days PLTR returned 15.36% versus SPY’s 28.10%, with significantly higher volatility and a deeper max drawdown. News flow has been constructive for the AI narrative—coverage highlights AI infrastructure positioning, improved earnings revisions, and defense-related interest (including drone-related focus and on-prem AI partnerships). At the same time, broader commentary warns about “AI mania,” potential pullbacks, and mixed short-term technicals, underscoring sentiment risk in a crowded theme.

Key uncertainties include the noisiness of news relative to future equity performance, the absence of peer-relative valuation and analyst estimate context in this dataset, and macro/sector factors that can drive high-beta software names irrespective of company-specific fundamentals. Policy and regulatory developments around AI oversight add another layer of uncertainty, even if not company-specific in this evidence.

Conclusion: The evidence supports a credible setup predicated on expanding revenue, strengthening margins, robust free cash flow, and a conservative balance sheet. However, elevated and expanding valuation multiples, recent underperformance versus SPY with higher volatility, and sentiment-dependent AI cycle risks meaningfully temper the case. On balance, PLTR has a plausible path to future outperformance if current fundamental momentum persists, but the setup is highly sensitive to sustaining growth and to market multiple stability in an AI-driven environment.

# 20. Final Discussion and Limitations

This notebook demonstrates a compact pattern for LLM-assisted equity research:

1. Fetch structured market and fundamental data.
2. Calculate deterministic evidence in Python.
3. Convert the evidence into JSON.
4. Ask the LLM to synthesize the already-computed evidence.

## Limitations

- FMP endpoint coverage may vary by subscription plan, ticker, exchange, or data category.
- Valuation ratios depend on FMP definitions and may differ from other providers.
- TTM calculations depend on available quarterly statement history and standardized FMP field names.
- News can be noisy, duplicated, incomplete, or only loosely connected to future stock performance.
- This is not financial advice.
- The LLM synthesizes evidence but does not independently verify facts beyond the provided evidence package.
- The notebook does not yet include peer comparison.
- The notebook does not include SEC filing RAG, earnings transcript RAG, or document retrieval.
- The notebook does not produce a valuation model, intrinsic value estimate, or price target.
- The notebook avoids buy/sell/hold language by design.

## Future Improvements

Potential extensions:

- Peer comparison.
- Sector-relative valuation.
- Analyst estimate trends.
- Earnings transcript analysis.
- SEC filing RAG, clearly separated from this deterministic evidence-routing workflow.
- Deterministic scoring rubric.
- More robust charting.
- PDF export.
- Interactive widgets for ticker selection and section toggles.
